[目录](./table_of_contents.ipynb)

# 扩展卡尔曼滤波器

In [ ]:
%matplotlib inline

In [ ]:
#格式化本书
import book_format
book_format.set_style()

我们已经为线性卡尔曼滤波器开发了理论。然后，在过去的两章中，我们涉及了使用卡尔曼滤波器解决非线性问题的主题。在本章中，我们将学习扩展卡尔曼滤波器（EKF）。EKF通过在线性化当前估计点处的系统，然后使用线性卡尔曼滤波器来过滤这个线性化系统来处理非线性。这是用于非线性问题的最早的技术之一，也是最常见的技术之一。

EKF对滤波器设计者提供了重大的数学挑战；这是本书中最具挑战性的一章。我尽我所能避免使用EKF，转而使用其他已经开发出来的用于过滤非线性问题的技术。然而，这个主题是不可避免的；所有经典论文和大多数当前领域的论文都使用EKF。即使您在自己的工作中不使用EKF，也需要熟悉这个主题才能阅读文献。

## 线性化卡尔曼滤波器

卡尔曼滤波器使用线性方程，因此无法处理非线性问题。问题可能有两种非线性方式。首先，过程模型可能是非线性的。物体穿过大气层时会受到减速的阻力。阻力系数根据物体的速度而变化。结果的行为是非线性的，不能用线性方程来建模。第二，测量可能是非线性的。例如，雷达给出目标的距离和方位。我们使用三角函数来计算目标的位置，这是非线性的。

对于线性滤波器，我们有以下过程和测量模型的方程式：

$$\begin{aligned}\dot{\mathbf x} &= \mathbf{Ax} + w_x\\
\mathbf z &= \mathbf{Hx} + w_z
\end{aligned}$$

其中，$\mathbf A$ 是系统动态矩阵。使用《卡尔曼滤波器数学》一章介绍的状态空间方法，这些方程可以转换为

$$\begin{aligned}\bar{\mathbf x} &= \mathbf{Fx} \\
\mathbf z &= \mathbf{Hx}
\end{aligned}$$

其中，$\mathbf F$ 是*基本矩阵*。噪声 $w_x$ 和 $w_z$ 术语被合并到矩阵 $\mathbf R$ 和 $\mathbf Q$ 中。这种形式的方程允许我们根据第 $k$ 步的测量和第 $k-1$ 步的状态估计计算状态。在之前的章节中，我使用描述牛顿方程的问题来建立了你的直觉并最小化了数学。我们知道如何根据高中物理设计 $\mathbf F$。

对于非线性模型，线性表达式 $\mathbf{Fx} + \mathbf{Bu}$ 被非线性函数 $f(\mathbf x, \mathbf u)$ 取代，线性表达式 $\mathbf{Hx}$ 被非线性函数 $h(\mathbf x)$ 取代：

$$\begin{aligned}\dot{\mathbf x} &= f(\mathbf x, \mathbf u) + w_x\\
\mathbf z &= h(\mathbf x) + w_z
\end{aligned}$$

可以想象，我们可以通过找到一组新的Kalman滤波器方程来最优地解决这些方程。但是，如果您记得“非线性滤波”一章中的图表，您将回想起通过非线性函数传递高斯分布会导致不再是高斯分布的概率分布。因此这样做行不通。

扩展卡尔曼滤波器不会改变卡尔曼滤波器的线性方程。相反，它在当前估计点处将非线性方程线性化，并在线性卡尔曼滤波器中使用这种线性化。

*线性化*的意思就像它听起来的那样。我们在定义点找到最接近曲线的线。下面的图表将在$x=1.5$处将抛物线$f(x)=x^2-2x$进行线性化。

In [ ]:
import kf_book.ekf_internal as ekf_internal
ekf_internal.show_linearization()

如果上面的曲线是过程模型，那么虚线显示了在估计$x=1.5$时该曲线的线性化。

通过求导来线性化系统，找到曲线的斜率：

$$\begin{aligned}
f(x) &= x^2 -2x \\
\frac{df}{dx} &= 2x - 2
\end{aligned}$$

然后在$x$处求值：

$$\begin{aligned}m &= f'(x=1.5) \\&= 2(1.5) - 2 \\&= 1\end{aligned}$$ 

线性化微分方程组也类似。通过对每个变量取偏导数来评估$\mathbf x_t$和$\mathbf u_t$处的$\mathbf F$和$\mathbf H$，我们线性化$f(\mathbf x, \mathbf u)$和$h(\mathbf x)$。矩阵的偏导数称为[*雅可比矩阵*](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant)。这给我们提供了离散状态转移矩阵和测量模型矩阵：

$$
\begin{aligned}
\mathbf F 
&= {\frac{\partial{f(\mathbf x_t, \mathbf u_t)}}{\partial{\mathbf x}}}\biggr|_{{\mathbf x_t},{\mathbf u_t}} \\
\mathbf H &= \frac{\partial{h(\bar{\mathbf x}_t)}}{\partial{\bar{\mathbf x}}}\biggr|_{\bar{\mathbf x}_t} 
\end{aligned}
$$

这导致了EKF的以下方程。我在与线性滤波器的不同之处周围放了框:

$$\begin{array}{l|l}
\text{线性卡尔曼滤波器} & \text{扩展卡尔曼滤波器（EKF）} \\
\hline 
& \boxed{\mathbf F = {\frac{\partial{f(\mathbf x_t, \mathbf u_t)}}{\partial{\mathbf x}}}\biggr|_{{\mathbf x_t},{\mathbf u_t}}} \\
\mathbf{\bar x} = \mathbf{Fx} + \mathbf{Bu} & \boxed{\mathbf{\bar x} = f(\mathbf x, \mathbf u)}  \\
\mathbf{\bar P} = \mathbf{FPF}^\mathsf{T}+\mathbf Q  & \mathbf{\bar P} = \mathbf{FPF}^\mathsf{T}+\mathbf Q \\
\hline
& \boxed{\mathbf H = \frac{\partial{h(\bar{\mathbf x}_t)}}{\partial{\bar{\mathbf x}}}\biggr|_{\bar{\mathbf x}_t}} \\
\textbf{y} = \mathbf z - \mathbf{H \bar{x}} & \textbf{y} = \mathbf z - \boxed{h(\bar{x})}\\
\mathbf{K} = \mathbf{\bar{P}H}^\mathsf{T} (\mathbf{H\bar{P}H}^\mathsf{T} + \mathbf R)^{-1} & \mathbf{K} = \mathbf{\bar{P}H}^\mathsf{T} (\mathbf{H\bar{P}H}^\mathsf{T} + \mathbf R)^{-1} \\
\mathbf x=\mathbf{\bar{x}} +\mathbf{K\textbf{y}} & \mathbf x=\mathbf{\bar{x}} +\mathbf{K\textbf{y}} \\
<End of Text>

\mathbf P= (\mathbf{I}-\mathbf{KH})\mathbf{\bar{P}} & \mathbf P= (\mathbf{I}-\mathbf{KH})\mathbf{\bar{P}}

通常我们不使用$\mathbf{Fx}$来传播EKF的状态，因为线性化会导致不准确性。通常使用适当的数值积分技术（如欧拉或龙格-库塔）计算$\bar{\mathbf x}$。因此，我写了$\mathbf{\bar x} = f(\mathbf x, \mathbf u)$。出于同样的原因，我们不在残差计算中使用$\mathbf{H\bar{x}}$，而选择更准确的$h(\bar{\mathbf x})$。

我认为理解EKF的最简单方法是从一个例子开始。稍后您可能希望回来重新阅读本节。

## 示例：跟踪飞机

这个例子使用基于地面的雷达来跟踪飞机。我们在上一章中为这个问题实现了一个UKF。现在我们将为同一个问题实现EKF，以便比较滤波器性能和实现滤波器所需的工作量。

雷达通过发射一束无线电波并扫描返回的反弹来工作。波束路径上的任何物体都会将信号的一部分反射回雷达。通过计算反射信号返回雷达所需的时间，系统可以计算出 *斜距离* - 从雷达安装到物体的直线距离。

雷达的斜距离 $r$ 和仰角 $\epsilon$ 与飞机的水平位置 $x$ 和高度 $y$ 之间的关系如下图所示：

In [ ]:
ekf_internal.show_radar_chart()

这给了我们以下相等式：

$$\begin{aligned}
\epsilon &= \tan^{-1} \frac y x\\
r^2 &= x^2 + y^2
\end{aligned}$$ 

### 设计状态变量

我们想要跟踪一架飞机的位置，假设它的速度和高度保持不变，并测量到飞机的斜距离。这意味着我们需要 3 个状态变量 - 水平距离、水平速度和高度：

$$\mathbf x = \begin{bmatrix}\mathtt{距离} \\\mathtt{速度}\\ \mathtt{高度}\end{bmatrix}=    \begin{bmatrix}x \\ \dot x\\ y\end{bmatrix}$$

### 设计过程模型

我们假设飞行器是一个牛顿运动学系统。我们在之前的章节中使用了这个模型，所以你可以通过检查发现我们想要：

$$\mathbf F = \left[\begin{array}{cc|c} 1 & \Delta t & 0\\
0 & 1 & 0 \\ \hline
0 & 0 & 1\end{array}\right]$$

我将矩阵分成块，以显示左上角块是$x$的恒定速度模型，右下角块是$y$的恒定位置模型。

然而，让我们练习找到这些矩阵。我们用一组微分方程来建模系统。我们需要一个形如

$$\dot{\mathbf x} = \mathbf{Ax} + \mathbf{w}$$
的方程，其中$\mathbf{w}$是系统噪声。

变量$x$和$y$是相互独立的，因此我们可以分开计算它们。一维运动的微分方程为：

$$\begin{aligned}v &= \dot x \\
a &= \ddot{x} = 0\end{aligned}$$

现在我们将微分方程转换为状态空间形式。如果这是一个二阶或更高阶微分系统，我们首先必须将它们缩小到等效的一阶方程组。这些方程是一阶的，因此我们将它们放入状态空间矩阵形式中，如下所示：

$$\begin{aligned}\begin{bmatrix}\dot x \\ \ddot{x}\end{bmatrix} &= \begin{bmatrix}0&1\\0&0\end{bmatrix} \begin{bmatrix}x \\ 
\dot x\end{bmatrix} \\ \dot{\mathbf x} &= \mathbf{Ax}\end{aligned}$$
其中 $\mathbf A=\begin{bmatrix}0&1\\0&0\end{bmatrix}$。

请注意，$\mathbf A$ 是*系统动态矩阵*。它描述了一组线性微分方程。我们必须从中计算出状态转移矩阵 $\mathbf F$。$\mathbf F$ 描述了一组离散的线性方程，用于计算离散时间步长 $\Delta t$ 中的 $\mathbf x$。

计算 $\mathbf F$ 的一种常见方法是使用矩阵指数的幂级数展开：

$$\mathbf F(\Delta t) = e^{\mathbf A\Delta t} = \mathbf{I} + \mathbf A\Delta t  + \frac{(\mathbf A\Delta t)^2}{2!} + \frac{(\mathbf A \Delta t)^3}{3!} + ... $$


由于 $\mathbf A^2 = \begin{bmatrix}0&0\\0&0\end{bmatrix}$，因此所有更高次幂的 $\mathbf A$ 都是 $\mathbf{0}$。因此幂级数展开为：

$$
\begin{aligned}
\mathbf F &=\mathbf{I} + \mathbf At + \mathbf{0} \\
&= \begin{bmatrix}1&0\\0&1\end{bmatrix} + \begin{bmatrix}0&1\\0&0\end{bmatrix}\Delta t\\
\mathbf F &= \begin{bmatrix}1&\Delta t\\0&1\end{bmatrix}
\end{aligned}$$

这与运动方程使用的结果相同！这个练习除了说明如何从线性微分方程中找到状态转移矩阵之外并没有什么必要。我们将用一个需要使用这种技术的例子来结束这一章。

### 设计测量模型

测量函数将先前状态的状态估计$\bar{\mathbf x}$转换为斜距离距离的测量值。我们使用勾股定理推导：

$$h(\bar{\mathbf x}) = \sqrt{x^2 + y^2}$$

由于平方根，斜距离与地面位置之间的关系是非线性的。我们通过在$\mathbf x_t$处评估其偏导数来线性化它：

$$
\mathbf H = \frac{\partial{h(\bar{\mathbf x})}}{\partial{\bar{\mathbf x}}}\biggr|_{\bar{\mathbf x}_t}
$$

矩阵的偏导数称为雅可比矩阵，其形式为

$$\frac{\partial \mathbf H}{\partial \bar{\mathbf x}} = 
\begin{bmatrix}
\frac{\partial h_1}{\partial x_1} & \frac{\partial h_1}{\partial x_2} &\dots \\
\frac{\partial h_2}{\partial x_1} & \frac{\partial h_2}{\partial x_2} &\dots \\
\vdots & \vdots
\end{bmatrix}
$$

换句话说，矩阵中的每个元素都是函数 $h$ 关于 $x$ 变量的偏导数。对于我们的问题，我们有

$$\mathbf H = \begin{bmatrix}{\partial h}/{\partial x} & {\partial h}/{\partial \dot{x}} & {\partial h}/{\partial y}\end{bmatrix}$$

依次求解：

$$\begin{aligned}
\frac{\partial h}{\partial x} &= \frac{\partial}{\partial x} \sqrt{x^2 + y^2} \\
&= \frac{x}{\sqrt{x^2 + y^2}}
\end{aligned}$$

以及

$$\begin{aligned}
\frac{\partial h}{\partial \dot{x}} &=
\frac{\partial}{\partial \dot{x}} \sqrt{x^2 + y^2} \\ 
&= 0
\end{aligned}$$

以及

$$\begin{aligned}
\frac{\partial h}{\partial y} &= \frac{\partial}{\partial y} \sqrt{x^2 + y^2} \\ 
&= \frac{y}{\sqrt{x^2 + y^2}}
\end{aligned}$$

从而得到

$$\mathbf H = 
\begin{bmatrix}
\frac{x}{\sqrt{x^2 + y^2}} & 
0 &
&
\frac{y}{\sqrt{x^2 + y^2}}
\end{bmatrix}$$

这可能看起来令人畏惧，所以退一步并认识到所有这些数学都在做一件非常简单的事情。我们有一个非线性的斜距方程，用于描述飞机的斜距。卡尔曼滤波器仅适用于线性方程，因此我们需要找到一个线性方程来近似 $\mathbf H$。如上所述，找到给定点上非线性方程的斜率是一个很好的近似值。对于卡尔曼滤波器，'给定点' 是状态变量 $\mathbf x$，因此我们需要对斜距进行关于 $\mathbf x$ 的导数。对于线性卡尔曼滤波器，$\mathbf H$ 是一个常数，我们在运行滤波器之前计算出它。对于扩展卡尔曼滤波器，随着评估点 $\bar{\mathbf x}$ 在每个时刻改变，$\mathbf H$ 在每个步骤中更新。

为了使这更具体化，现在让我们编写一个 Python 函数来计算此问题的 $h$ 的雅各比矩阵。

In [ ]:
from math import sqrt
def HJacobian_at(x):
    """计算 x 处 H 矩阵的雅各比矩阵"""

In [ ]:
horiz_dist = x[0]
    altitude   = x[2]
    denom = sqrt(horiz_dist**2 + altitude**2)
    return array ([[horiz_dist/denom, 0., altitude/denom]])

最后，让我们提供 $h(\bar{\mathbf x})$ 的代码：

In [ ]:
def hx(x):
    """计算对应于状态 x 的斜距测量值。"""
    
    return (x[0]**2 + x[2]**2) ** 0.5

现在，让我们为我们的雷达编写一个模拟。

In [ ]:
from numpy.random import randn
import math

In [ ]:
class RadarSim:
    """ 模拟一个物体在一维空间上以恒定高度和速度飞行时的雷达信号返回。 """
    
    def __init__(self, dt, pos, vel, alt):
        self.pos = pos
        self.vel = vel
        self.alt = alt
        self.dt = dt
        
    def get_range(self):
        """ 返回到物体的斜距离。每次在上次调用后的dt时间调用一次。 """
        
        # 在系统中添加一些过程噪声
        self.vel = self.vel  + .1*randn() # randn()表示正态分布noise
        self.alt = self.alt + .1*randn()
        self.pos = self.pos + self.vel*self.dt
    
        # 添加测量噪声
        err = self.pos * 0.05*randn() # randn()表示正态分布noise
        slant_dist = math.sqrt(self.pos**2 + self.alt**2)
        
        return slant_dist + err

### 设计过程和测量噪声

雷达测量目标的距离。我们将使用 $\sigma_{range}= 5$ 米作为噪声。这给了我们

$$\mathbf R = \begin{bmatrix}\sigma_{range}^2\end{bmatrix} = \begin{bmatrix}25\end{bmatrix}$$


需要讨论设计 $\mathbf Q$。状态 $\mathbf x= \begin{bmatrix}x & \dot x & y\end{bmatrix}^\mathtt{T}$。前两个元素是位置（下行距离）和速度，所以我们可以使用 `Q_discrete_white_noise` 噪声来计算 $\mathbf Q$ 左上角的值。$\mathbf x$ 的第三个元素是高度，我们假设它与下行距离独立。这导致我们将 $\mathbf Q$ 分成块：

$$\mathbf Q = \begin{bmatrix}\mathbf Q_\mathtt{x} & 0 \\ 0 & \mathbf Q_\mathtt{y}\end{bmatrix}$$

### 实现

`FilterPy` 提供了 `ExtendedKalmanFilter` 类。它的工作方式类似于我们一直在使用的 `KalmanFilter` 类，但它允许您提供一个计算 $\mathbf H$ 的雅可比矩阵和函数 $h(\mathbf x)$ 的函数。

我们首先导入滤波器并创建它。 `x` 的维度为 3，`z` 的维度为 1。

In [ ]:
from filterpy.kalman import ExtendedKalmanFilter

rk = ExtendedKalmanFilter(dim_x=3, dim_z=1)

我们创建雷达模拟器：

In [ ]:
radar = RadarSim(dt, pos=0., vel=100., alt=1000.)

我们将在飞机的实际位置附近初始化滤波器：

In [ ]:
rk.x = array([radar.pos, radar.vel-10, radar.alt+100])

我们使用上面计算的泰勒级数展开式的第一项来分配系统矩阵：

In [ ]:
dt = 0.05
rk.F = eye(3) + array([[0, 1, 0],
                       [0, 0, 0],
                       [0, 0, 0]])*dt

在为 $\mathbf R$、$\mathbf Q$ 和 $\mathbf P$ 分配合理的值后，我们可以通过简单的循环运行滤波器。我们将计算 $\mathbf H$ 和 $h(x)$ 的雅可比函数传递到 `update` 方法中。

In [ ]:
for i in range(int(20/dt)):
    z = radar.get_range()
    rk.update(array([z]), HJacobian_at, hx)
    rk.predict()

添加一些样板代码以保存和绘制结果：

In [ ]:
from filterpy.common import Q_discrete_white_noise
from filterpy.kalman import ExtendedKalmanFilter
from numpy import eye, array, asarray
import numpy as np

dt = 0.05
rk = ExtendedKalmanFilter(dim_x=3, dim_z=1)
radar = RadarSim(dt, pos=0., vel=100., alt=1000.)

# 做出一个不完美的初始猜测
rk.x = array([radar.pos-100, radar.vel+100, radar.alt+1000])

rk.F = eye(3) + array([[0, 1, 0],
                       [0, 0, 0],
                       [0, 0, 0]]) * dt

range_std = 5. # 米
rk.R = np.diag([range_std**2])
rk.Q[0:2, 0:2] = Q_discrete_white_noise(2, dt=dt, var=0.1)
rk.Q[2,2] = 0.1
rk.P *= 50

In [ ]:
xs, track = [], []
for i in range(int(20/dt)):
    z = radar.get_range()
    track.append((radar.pos, radar.vel, radar.alt))
    
    rk.update(array([z]), HJacobian_at, hx)
    xs.append(rk.x)
    rk.predict()

xs = asarray(xs)
track = asarray(track)
time = np.arange(0, len(xs)*dt, dt)
ekf_internal.plot_radar(xs, track, time)

## 使用 SymPy 计算 Jacobian 矩阵

根据您对导数的经验，您可能会发现 Jacobian 矩阵的计算很困难。即使您觉得很简单，稍微复杂一点的问题也很容易导致非常困难的计算。

正如附录 A 中所述，我们可以使用 SymPy 包来为我们计算 Jacobian 矩阵。

In [ ]:
import sympy
from IPython.display import display
sympy.init_printing(use_latex='mathjax')

x, x_vel, y = sympy.symbols('x, x_vel y')

H = sympy.Matrix([sympy.sqrt(x**2 + y**2)])

state = sympy.Matrix([x, x_vel, y])
J = H.jacobian(state)

display(state)
display(J)

这个结果和我们之前计算的结果一样，但是我们花费的力气要少得多！

## 机器人定位

现在是尝试一个真实问题的时候了。我告诉你，这一节很难。然而，大多数书籍选择简单的、有简单答案的教材问题，而你却不知道如何解决真实世界的问题。

我们将考虑机器人定位问题。我们已经在**无迹卡尔曼滤波器**章节中实现了这个问题，如果你还没有阅读过，我建议你现在就去看。在这种情况下，我们有一个机器人通过传感器检测地标的移动。这可能是一辆使用计算机视觉识别树木、建筑和其他地标的自动驾驶汽车。它可能是那些吸尘你家的小型机器人之一，或者是仓库里的机器人。

机器人有四个轮子，采用汽车相同的配置。 它通过旋转前轮来操纵。 这会导致机器人在向前移动时围绕后轴旋转。 这是非线性行为，我们必须对其进行建模。

机器人具有测量景观中已知目标的范围和方位角的传感器。 这是非线性的，因为从范围和方位角计算位置需要平方根和三角函数。

过程模型和测量模型都是非线性的。 EKF同时适应两者，因此我们暂时得出结论：EKF是解决此问题的可行选择。

### 机器人运动模型

在第一近似下，汽车在前轮向前旋转时转向。汽车的前部沿着轮子所指的方向移动，而围绕后轮旋转。由于摩擦引起的滑移、不同速度下橡胶轮胎的不同行为以及外侧轮胎需要行驶不同的半径，这个简单的描述变得复杂了。准确地建模转向需要一组复杂的微分方程。

在低速机器人应用中，已经发现了一个更简单的“自行车模型”，它表现良好。这是该模型的描述：

In [ ]:
ekf_internal.plot_bicycle()

在**无迹卡尔曼滤波器**一章中，我们推导出了这些方程：

$$\begin{aligned} 
\beta &= \frac d w \tan(\alpha) \\
x &= x - R\sin(\theta) + R\sin(\theta + \beta) \\
y &= y + R\cos(\theta) - R\cos(\theta + \beta) \\
\theta &= \theta + \beta
\end{aligned}
$$

其中$\theta$是机器人的航向。

如果您对转向模型不感兴趣，则不需要详细了解此模型。重要的是要认识到我们的运动模型是非线性的，因此我们需要使用我们的卡尔曼滤波器来处理它。

### 设计状态变量

对于我们的滤波器，我们将保持机器人的位置$x,y$和方向$\theta$：

$$\mathbf x = \begin{bmatrix}x \\ y \\ \theta\end{bmatrix}$$

我们的控制输入$\mathbf u$是速度$v$和转向角$\alpha$：

$$\mathbf u = \begin{bmatrix}v \\ \alpha\end{bmatrix}$$

### 设计系统模型

我们将我们的系统建模为非线性运动模型加上噪声。

$$\bar x = f(x, u) + \mathcal{N}(0, Q)$$

使用我们上面创建的机器人运动模型，我们可以将其扩展到

$$\bar{\begin{bmatrix}x\\y\\\theta\end{bmatrix}} = \begin{bmatrix}x\\y\\\theta\end{bmatrix} + 
\begin{bmatrix}- R\sin(\theta) + R\sin(\theta + \beta) \\
R\cos(\theta) - R\cos(\theta + \beta) \\
\beta\end{bmatrix}$$

通过对 $f(x,u)$ 的雅可比矩阵，我们可以找到 $\mathbf F$。

$$\mathbf F = \frac{\partial f(x, u)}{\partial x} =\begin{bmatrix}
\frac{\partial f_1}{\partial x} & 
\frac{\partial f_1}{\partial y} &
\frac{\partial f_1}{\partial \theta}\\
\frac{\partial f_2}{\partial x} & 
\frac{\partial f_2}{\partial y} &
\frac{\partial f_2}{\partial \theta} \\
\frac{\partial f_3}{\partial x} & 
\frac{\partial f_3}{\partial y} &
\frac{\partial f_3}{\partial \theta}
\end{bmatrix}
$$

当我们计算它们时，得到

$$\mathbf F = \begin{bmatrix}
1 & 0 & -R\cos(\theta) + R\cos(\theta+\beta) \\
0 & 1 & -R\sin(\theta) + R\sin(\theta+\beta) \\
0 & 0 & 1
\end{bmatrix}$$

我们可以用 SymPy 再次检查我们的工作。

In [ ]:
import sympy
from sympy.abc import alpha, x, y, v, w, R, theta
from sympy import symbols, Matrix
sympy.init_printing(use_latex="mathjax", fontsize='16pt')
time = symbols('t')
d = v*time
beta = (d/w)*sympy.tan(alpha)
r = w/sympy.tan(alpha)

fxu = Matrix([[x-r*sympy.sin(theta) + r*sympy.sin(theta+beta)],
              [y+r*sympy.cos(theta)- r*sympy.cos(theta+beta)],
              [theta+beta]])
F = fxu.jacobian(Matrix([x, y, theta]))
F

看起来有些复杂。我们可以使用 SymPy 替换一些术语：

In [ ]:
# reduce common expressions
B, R = symbols('beta, R')
F = F.subs((d/w)*sympy.tan(alpha), B)
F.subs(w/sympy.tan(alpha), R)

这种形式验证了 Jacobian 的计算是正确的。

现在我们可以把注意力转向噪声。在这里，噪声是在我们的控制输入中，因此它在*控制空间*中。换句话说，我们命令一个特定的速度和转向角度，但我们需要将其转换为$x，y，\theta$中的误差。在实际系统中，这可能会因速度而异，因此需要为每个预测重新计算。我将选择此作为噪声模型；对于真实的机器人，您需要选择一个准确描述系统误差的模型。

$$\mathbf{M} = \begin{bmatrix}\sigma_{vel}^2 & 0 \\ 0 & \sigma_\alpha^2\end{bmatrix}$$

如果这是一个线性问题，我们将使用现在熟悉的$\mathbf{FMF}^\mathsf T$形式从控制空间转换为状态空间。由于我们的运动模型是非线性的，因此我们不尝试找到闭合形式的解，而是使用Jacobian将其线性化，我们将其命名为$\mathbf{V}$。

$$\mathbf{V} = \frac{\partial f(x, u)}{\partial u} \begin{bmatrix}
\frac{\partial f_1}{\partial v} & \frac{\partial f_1}{\partial \alpha} \\
\frac{\partial f_2}{\partial v} & \frac{\partial f_2}{\partial \alpha} \\
\frac{\partial f_3}{\partial v} & \frac{\partial f_3}{\partial \alpha}
\end{bmatrix}$$

这些偏导数变得非常难处理。让我们用SymPy计算它们。

In [ ]:
V = fxu.jacobian(Matrix([v, alpha]))
V = V.subs(sympy.tan(alpha)/w, 1/R) 
V = V.subs(time*v/R, B)
V = V.subs(time*v, 'd')
V

这应该让您感受到EKF变得数学上难以处理的速度。

这给我们带来了预测方程的最终形式：

$$\begin{aligned}
\mathbf{\bar x} &= \mathbf x + 
\begin{bmatrix}- R\sin(\theta) + R\sin(\theta + \beta) \\
R\cos(\theta) - R\cos(\theta + \beta) \\
\beta\end{bmatrix}\\
\mathbf{\bar P} &=\mathbf{FPF}^{\mathsf T} + \mathbf{VMV}^{\mathsf T}
\end{aligned}$$

这种线性化不是预测 $\mathbf x$ 的唯一方法。例如，我们可以使用数值积分技术，如 *Runge Kutta* 来计算机器人的运动。如果时间步长相对较大，则需要这样做。与 Kalman 滤波器不同，EKF 的情况不是那么简单。对于一个真实的问题，你必须仔细地用微分方程对系统进行建模，然后确定解决该系统的最适当的方法。正确的方法取决于所需的精度、方程的非线性程度、处理器预算和数值稳定性问题。

### 设计测量模型

机器人的传感器提供了对景观中多个已知位置的噪声承受和距离测量。测量模型必须将状态 $\begin{bmatrix}x & y&\theta\end{bmatrix}^\mathsf T$ 转换为到地标的距离和承受。如果 $\mathbf p$ 是地标的位置，则范围 $r$ 是：

$$r = \sqrt{(p_x - x)^2 + (p_y - y)^2}$$

传感器提供相对于机器人方向的方位角，因此我们必须从方位角中减去机器人的方向以获取传感器读数，如下所示：

$$\phi = \arctan(\frac{p_y - y}{p_x - x}) - \theta$$

因此，我们的测量模型 $h$ 是

$$\begin{aligned}
\mathbf z& = h(\bar{\mathbf x}, \mathbf p) &+ \mathcal{N}(0, R)\\
&= \begin{bmatrix}
\sqrt{(p_x - x)^2 + (p_y - y)^2} \\
\arctan(\frac{p_y - y}{p_x - x}) - \theta 
\end{bmatrix} &+ \mathcal{N}(0, R)
\end{aligned}$$

这显然是非线性的，因此我们需要在线性化 $\mathbf x$ 处对 $h$ 进行线性化，即通过取其雅可比矩阵来计算。我们下面使用 SymPy 计算。

In [ ]:
px, py = symbols('p_x, p_y')
z = Matrix([[sympy.sqrt((px-x)**2 + (py-y)**2)],
            [sympy.atan2(py-y, px-x) - theta]])
z.jacobian(Matrix([x, y, theta]))

现在我们需要将其编写为 Python 函数。例如，我们可以这样写：

In [ ]:
from math import sqrt
<End of Text>```


def H_of(x, landmark_pos):
    """计算H矩阵的雅各比矩阵，其中h(x)计算状态x与地标的距离和方位角"""

In [ ]:
px = landmark_pos[0]
py = landmark_pos[1]
hyp = (px - x[0, 0])**2 + (py - x[1, 0])**2
dist = sqrt(hyp)

H = array(
    [[-(px - x[0, 0]) / dist, -(py - x[1, 0]) / dist, 0],
     [ (py - x[1, 0]) / hyp,  -(px - x[0, 0]) / hyp, -1]])
return H
```

我们还需要定义一个函数，将系统状态转换为测量值。

In [ ]:
from math import atan2

def Hx(x, landmark_pos):
    """获取状态变量并返回与该状态相对应的测量值。"""

    px = landmark_pos[0]
    py = landmark_pos[1]
    dist = sqrt((px - x[0, 0])**2 + (py - x[1, 0])**2)

    Hx = array([[dist],
                [atan2(py - x[1, 0], px - x[0, 0]) - x[2, 0]]])
    return Hx

### 设计测量噪声

可以合理地假设测量的距离和方位角的噪声是独立的，因此

$$\mathbf R=\begin{bmatrix}\sigma_{range}^2 & 0 \\ 0 & \sigma_{bearing}^2\end{bmatrix}$$

### 实现

我们将使用 `FilterPy` 的 `ExtendedKalmanFilter` 类来实现滤波器。它的 `predict()` 方法使用标准线性方程来处理过程模型。而我们的模型是非线性的，因此我们将覆盖 `predict()` 方法并使用我们自己的实现。我还想使用这个类来模拟机器人，因此我将添加一个 `move()` 方法，该方法计算机器人的位置，`predict()` 方法和模拟都可以调用。

预测步骤的矩阵非常大。在编写此代码时，我在最终使其正常工作之前犯了几个错误。我只有通过使用SymPy的 `evalf` 函数才找到了我的错误。`evalf` 使用变量的特定值来计算SymPy `Matrix`。我决定向您演示这个技巧，并在卡尔曼滤波器代码中使用了 `evalf`。您需要了解几个要点。

首先，`evalf` 使用字典来指定值。例如，如果您的矩阵包含 `x` 和 `y`，您可以编写

In [ ]:
    M.evalf(subs={x:3, y:17})

来计算 `x=3` 和 `y=17` 的矩阵。

其次，`evalf` 返回一个 `sympy.Matrix` 对象。使用 `numpy.array(M).astype(float)` 将其转换为 NumPy 数组。`numpy.array(M)` 创建了一个 `object` 类型的数组，这不是您想要的。

以下是EKF的代码：

In [ ]:
from filterpy.kalman import ExtendedKalmanFilter as EKF
from numpy import array, sqrt
class RobotEKF(EKF):
    def __init__(self, dt, wheelbase, std_vel, std_steer):
        EKF.__init__(self, 3, 2, 2)
        self.dt = dt
        self.wheelbase = wheelbase
        self.std_vel = std_vel
        self.std_steer = std_steer

        a, x, y, v, w, theta, time = symbols(
            'a, x, y, v, w, theta, t')
        d = v*time
        beta = (d/w)*tan(a)
        r = w/tan(a)
    
        self.fxu = Matrix(
            [[x-r*sin(theta)+r*sin(theta+beta)],
             [y+r*cos(theta)-r*cos(theta+beta)],
             [theta+beta]])

        self.F_j = self.fxu.jacobian(Matrix([x, y, theta]))
        self.V_j = self.fxu.jacobian(Matrix([v, a]))

# 保存字典及其变量以备将来使用
        self.subs = {x: 0, y: 0, v:0, a:0, 
                     time:dt, w:wheelbase, theta:0}
        self.x_x, self.x_y, = x, y 
        self.v, self.a, self.theta = v, a, theta

In [ ]:
def predict(self, u):
    self.x = self.move(self.x, u, self.dt)
    self.subs[self.x_x] = self.x[0, 0]
    self.subs[self.x_y] = self.x[1, 0]

    self.subs[self.theta] = self.x[2, 0]
    self.subs[self.v] = u[0]
    self.subs[self.a] = u[1]

    F = array(self.F_j.evalf(subs=self.subs)).astype(float)
    V = array(self.V_j.evalf(subs=self.subs)).astype(float)

    # 在控制空间中运动噪声的协方差
    M = array([[self.std_vel**2, 0], 
               [0, self.std_steer**2]])

    self.P = F @ self.P @ F.T + V @ M @ V.T

def move(self, x, u, dt):
    hdg = x[2, 0]
    vel = u[0]
    steering_angle = u[1]
    dist = vel * dt

如果绝对值(转向角度) > 0.001: # 机器人是否在转弯？
            beta = (dist / 自身轴距) * tan(转向角度)
            r = 自身轴距 / tan(转向角度) # 半径

In [ ]:
dx = np.array([[-r*sin(hdg) + r*sin(hdg + beta)], 
               [r*cos(hdg) - r*cos(hdg + beta)], 
               [beta]])
        else: # 直线行驶
dx = np.array([[dist*cos(hdg)], 
               [dist*sin(hdg)], 
               [0]])
        return x + dx
```

现在我们有另一个问题要处理。残差概念上是计算为 $y = z - h(x)$，但这样做行不通，因为我们的测量中包含一个角度。假设 z 的方位角为 $1^\circ$，$h(x)$ 的方位角为 $359^\circ$，简单相减会得到一个角度差为 $-358^\circ$，而正确的值应该是 $2^\circ$。我们必须编写代码来正确计算方位角残差。

In [ ]:
def residual(a, b):
    """在包含[距离，方位]的测量之间计算residual（a-b）。
    方位角归一化为[-pi，pi)"""
    y = a - b
    y[1] = y[1] % (2 * np.pi)  #强制限制在[0,2 pi)范围内
    if y[1] > np.pi:  #移动到[-pi, pi)范围内
        y[1] -= 2 * np.pi
    return y

其余的代码运行模拟并绘制结果，现在不需要太多注释。
我创建了一个包含地标坐标的变量“landmarks”。
我每秒更新模拟机器人的位置10次，但每秒仅运行一次EKF。
这是由于两个原因。首先，我们没有使用龙格库塔法（Runge Kutta）来积分运动微分方程，
因此狭窄的时间步长使得我们的模拟更加准确。第二，
在嵌入式系统中，有限的处理速度是相当普遍的。这迫使您仅在绝对必要的时候运行KalmanFilter。

```python
from filterpy.stats import plot_covariance_ellipse
from math import sqrt, tan, cos, sin, atan2
import matplotlib.pyplot as plt

dt = 1.0

def z_landmark(lmark, sim_pos, std_rng, std_brg):
    x, y = sim_pos[0, 0], sim_pos[1, 0]
    d = np.sqrt((lmark[0] - x)**2 + (lmark[1] - y)**2)  
    a = atan2(lmark[1] - y, lmark[0] - x) - sim_pos[2, 0]
    z = np.array([[d + randn()*std_rng],
                  [a + randn()*std_brg]])
    return z

In [ ]:
def ekf_update(ekf, z, landmark):
    ekf.update(z, HJacobian=H_of, Hx=Hx, 
               residual=residual,
               args=(landmark), hx_args=(landmark))
    
                
def run_localization(landmarks, std_vel, std_steer, 
                     std_range, std_bearing,
                     step=10, ellipse_step=20, ylim=None):
    ekf = RobotEKF(dt, wheelbase=0.5, std_vel=std_vel, 
                   std_steer=std_steer)
    ekf.x = array([[2, 6, .3]]).T # x, y, 方向盘角度
    ekf.P = np.diag([.1, .1, .1])
    ekf.R = np.diag([std_range**2, std_bearing**2])

    sim_pos = ekf.x.copy() # 模拟位置
    # 方向盘指令 (速度, 方向盘角度弧度)
    u = array([1.1, .01]) 

    plt.figure()
    plt.scatter(landmarks[:, 0], landmarks[:, 1],
                marker='s', s=60)
    
    track = []
    for i in range(200):
        sim_pos = ekf.move(sim_pos, u, dt/10.) # 模拟机器人
        track.append(sim_pos)

如果 i % 步长 == 0：
            ekf.predict(u=u)

In [ ]:
如果 i % 椭圆步长 == 0：
    plot_covariance_ellipse(
        (ekf.x[0,0], ekf.x[1,0]), ekf.P[0:2, 0:2], 
         std=6, facecolor='k', alpha=0.3)

x, y = sim_pos[0, 0], sim_pos[1, 0]
for lmark in landmarks:
    z = z_landmark(lmark, sim_pos,
                   std_range, std_bearing)
    ekf_update(ekf, z, lmark)

如果 i % 椭圆步长 == 0：
    plot_covariance_ellipse(
        (ekf.x[0,0], ekf.x[1,0]), ekf.P[0:2, 0:2],
        std=6, facecolor='g', alpha=0.8)
    track = np.array(track)
    plt.plot(track[:, 0], track[:,1], color='k', lw=2)
    plt.axis('equal')
    plt.title("EKF 机器人定位")
    如果 ylim 不为 None： plt.ylim(*ylim)
    plt.show()
    return ekf

In [ ]:
landmarks = array([[5, 10], [10, 5], [15, 15]])


ekf = run_localization(
    landmarks, std_vel=0.1, std_steer=np.radians(1),
    std_range=0.3, std_bearing=0.1)
print('最终 P 值:', ekf.P.diagonal())

In [ ]:

我已经将地标绘制为实心正方形。机器人的路径用黑线绘制。预测步骤的协方差椭圆为浅灰色，更新的协方差为绿色。为了使它们在这个比例尺下可见，我将椭圆边界设置为6 $sigma$。

我们可以看到，我们的运动模型增加了很多不确定性，大部分误差在运动方向上。我们从蓝色椭圆的形状中确定这一点。几步之后，我们可以看到滤波器结合了地标测量，误差得到了改善。

我在 UKF 章节中使用了相同的初始条件和地标位置。UKF 在误差椭圆方面的精度更高。在估计 $\mathbf x$ 方面，两者的表现大致相同。

现在让我们添加另一个地标。

```python
landmarks = array([[5, 10], [10, 5], [15, 15], [20, 5]])

ekf = run_localization(
    landmarks, std_vel=0.1, std_steer=np.radians(1),
    std_range=0.3, std_bearing=0.1)
plt.show()
print('最终 P:', ekf.P.diagonal())

在跑道结束附近的估计不确定性更小。我们可以通过只使用前两个地标来看到多个地标对我们的不确定性产生的影响。

In [ ]:
ekf = run_localization(
    landmarks[0:2], std_vel=1.e-10, std_steer=1.e-10,
    std_range=1.4, std_bearing=.05)
print('最终 P:', ekf.P.diagonal())

经过地标后，估计很快偏离了机器人的路径。方差也快速增长。让我们看看只有一个地标时会发生什么：

In [ ]:
ekf = run_localization(
    landmarks[0:1], std_vel=1.e-10, std_steer=1.e-10,
    std_range=1.4, std_bearing=.05)
print('最终 P:', ekf.P.diagonal())

正如你所猜测的那样，一个地标会产生非常糟糕的结果。相反，大量的地标可以让我们做出非常准确的估计。

In [ ]:
landmarks = array([[5, 10], [10,  5], [15, 15], [20,  5], [15, 10], 
                   [10,14], [23, 14], [25, 20], [10, 20]])

ekf = run_localization(
    landmarks, std_vel=0.1, std_steer=np.radians(1),
    std_range=0.3, std_bearing=0.1, ylim=(0, 21))
print('Final P:', ekf.P.diagonal())

### 讨论

我说这是一个真正的问题，在某些方面确实是这样。我见过使用机器人运动模型的替代演示文稿，从而导致了更简单的雅可比矩阵。另一方面，我的运动模型在几个方面也很简单。首先，它使用自行车模型。真实的汽车有两组轮胎，每个轮胎行驶的半径不同。车轮不能完美地抓住路面。我还假设机器人对控制输入的响应是瞬时的。Sebastian Thrun在《概率机器人》中写道，当这种简化模型用于跟踪真实车辆时，滤波器的表现良好，这种简化模型是有道理的。这里的教训是，虽然必须具有合理准确的非线性模型，但它不需要完美才能运作良好。作为设计师，您需要在模型的保真度、数学难度和执行线性代数所需的CPU时间之间进行平衡。

这个问题过于简单化的另一个方面是我们假设我们知道地标和测量之间的对应关系。但是，假设我们正在使用雷达，我们如何知道特定信号返回对应于局部场景中的特定建筑物？这个问题暗示了SLAM算法 - 同时定位和建图。SLAM不是本书的重点，因此我不会详细阐述这个主题。

## UKF vs EKF

在上一章中，我使用了UKF来解决这个问题。实现的差异应该非常明显。即使有一个基本的运动模型，计算状态和测量模型的雅可比矩阵也并不容易。不同的问题可能导致一个难以或不可能解析求导的雅可比矩阵。相比之下，UKF只需要您提供一个计算系统运动模型的函数和另一个计算测量模型的函数。

有很多情况下雅可比矩阵无法通过解析方法求得。这些细节超出了本书的范围，但你将不得不使用数值方法来计算雅可比矩阵。这项工作并不容易，你将在STEM学校的硕士学位中花费大量时间学习处理这类情况的技术。即使如此，你也只能解决与你领域相关的问题——航空工程师学习了很多关于纳维-斯托克斯方程的知识，但对于建模化学反应速率方面的研究并不多。 

因此，UKF很容易，但它们准确吗？实践中，它们通常比EKF表现更好。你可以找到很多研究论文证明UKF在各种问题领域中优于EKF。很容易理解为什么会这样，EKF通过在线性化系统模型和测量模型的单个点上工作，而UKF使用$2n+1$个点。

让我们来看一个具体的例子。取 $f(x) = x^3$ 并通过高斯分布传递它。我将使用蒙特卡罗模拟计算一个精确的答案。我随机生成 50000 个点，这些点是根据高斯分布随机分布的，将每个点通过 $f(x)$，然后计算结果的均值和方差。

EKF 通过取导数来线性化函数，以找到评估点 $x$ 的斜率。此斜率成为我们用于转换高斯分布的线性函数。下面是该图的绘图。

In [ ]:
import kf_book.nonlinear_plots as nonlinear_plots
nonlinear_plots.plot_ekf_vs_mc()

EKF 计算相当不准确。相比之下，以下是 UKF 的性能：

In [ ]:
nonlinear_plots.plot_ukf_vs_mc(alpha=0.001, beta=3., kappa=1.)

这里我们可以看到，UKF的均值计算精确到小数点后2位。标准差略有偏差，但是您还可以使用生成sigma点的$\alpha$，$\beta$和$\gamma$参数微调UKF计算分布的方式。在这里，我使用了$\alpha=0.001$，$\beta=3$和$\gamma=1$。请随意修改它们以查看结果。您应该能够比我获得更好的结果。但是，避免为特定测试过度调整UKF。它可能在您的测试用例中表现更好，但在一般情况下表现更差。